## Import Libraries

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from string import punctuation
import numpy as np
from sklearn import preprocessing

## Load data 

In [ ]:
path= r"E:\Machine Learning\data\Spam_data.csv"

spam_data=pd.read_csv(path)

In [ ]:
spam_data.head()

In [ ]:
spam_data.count()

## Separate Input Variable (also called independent variable (X)) and dependent variable (y)

In [ ]:
X,y = spam_data['Message'].to_frame(),spam_data['Category'].to_frame()

In [ ]:
X.head()

In [ ]:
X.count()

In [ ]:
y.head()

In [ ]:
y.count()

## Text Preprocessing Pipeline

In [ ]:
lemmatizer = WordNetLemmatizer()

def pre_process_message(message):
    
    # tokenize the message
    words = nltk.word_tokenize(message)
    
    # remove stopwords
    cleaned_words = [word for word in words if word not in set(stopwords.words('english'))]
    
    # remove special characters
    filtered_words = [word for word in cleaned_words if word not in set(punctuation)]
    
    # remove words with length<=3
    
    pruned_words = [word for word in filtered_words if len(words)>3]
    
    # lemmatize the tokens
    
    lemmas = [lemmatizer.lemmatize(word) for word in pruned_words]
    
    return ' '.join(lemmas)


#message='spam classifier is a huge pipeline. The pipeline identification number is 23456. We need to complete it tomorrow'

#pre_process_message(message)

X['processed_sentences']=X['Message'].map(pre_process_message)
    

In [ ]:
X.count()

In [ ]:
y.count()

## Split the data into training and test sets.


X_train, y_train will be used for training the model.
X_test will be used for calculating predictions on unseen data.
y_test will be used to evaluate the model.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X['processed_sentences'],y,test_size=0.3,random_state=0)

## Feature Engineering

### 1. Convert processed sentences to vectors

In [ ]:


tfidf_vect = TfidfVectorizer(max_features=1500,min_df=5,max_df=0.7)
tfidf_model=tfidf_vect.fit(X['processed_sentences'])


X_train_vectors = tfidf_model.transform(X_train)
X_test_vectors = tfidf_model.transform(X_test)


In [ ]:
X_train_vectors

### 2. Convert class labels to indexes

In [ ]:
le = preprocessing.LabelEncoder()
le.fit(y)

y_train_indexes = le.transform(y_train)
y_test_indexed = le.transform(y_test)

In [ ]:
le.classes_

In [ ]:
le.inverse_transform([0,1])

In [ ]:
y_train_indexes

## Model Building

In [ ]:
# We will use RandomForrest Classifier

classifier = RandomForestClassifier(n_estimators=100, random_state=0)

classifier = classifier.fit(X_train_vectors,y_train_indexes)

In [ ]:

# run predictions on unseen data

y_pred=classifier.predict(X_test_vectors)

## Evaluate the model

In [ ]:
print(confusion_matrix(y_test_indexed,y_pred))  
print(classification_report(y_test_indexed,y_pred))  
print(accuracy_score(y_test_indexed, y_pred)) 

## Save the models

1. Vectorizer

In [ ]:
with open(r'E:\Personal\ML_Journey\model\vectorizer.pkl', 'wb') as model:  
    pickle.dump(tfidf_model,model)

2. Classifier

In [ ]:
with open(r'E:\Personal\ML_Journey\model\spam_classifier.pkl', 'wb') as model:  
    pickle.dump(classifier,model)

## Load Models and test them.

In [ ]:
# we use pickel library

import pickle

vectorizer_path=r'E:\Personal\ML_Journey\model\vectorizer.pkl'
classifier_path=r'E:\Personal\ML_Journey\model\spam_classifier.pkl'


vectorizer = pickle.load(open(vectorizer_path, 'rb'))

classifier = pickle.load(open(classifier_path, 'rb'))

In [ ]:
processed_text = pre_process_message("Exciting offer!!!! Hurray you have won 10000 dollars. Click here to claim your reward.")

In [ ]:
processed_text

In [ ]:
vectorized_msg = vectorizer.transform([processed_text])

In [ ]:
vectorized_msg

In [ ]:
prediction = classifier.predict(vectorized_msg)

In [ ]:
prediction